In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from PIL import Image
from tqdm import tqdm
import copy
import matplotlib.pyplot as plt

In [3]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/baselineModels/deeplabv3'

In [4]:
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images")
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images'

In [6]:
from accuracy_indian import compute_metrics

In [7]:
class SegmentationDataset(Dataset):
    def __init__(self, root, image_folder="images", mask_folder="masks", transforms=None):
        self.root = root
        self.transforms = transforms
        self.image_folder = os.path.join(root, image_folder)
        self.mask_folder = os.path.join(root, mask_folder)
        self.image_names = sorted(os.listdir(self.image_folder))
        self.mask_names = sorted(os.listdir(self.mask_folder))

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_folder, self.image_names[idx])
        mask_path = os.path.join(self.mask_folder, self.mask_names[idx])
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Grayscale mask
        if self.transforms:
            image = self.transforms(image)
            mask = transforms.ToTensor()(mask)  # Mask to tensor (0-1 range)
        return {"image": image, "mask": mask}

In [8]:
def iou_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return intersection / (union + 1e-10) if union > 0 else 1.0

def f1_score(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    tp = np.sum(y_true * y_pred)
    fp = np.sum(y_pred) - tp
    fn = np.sum(y_true) - tp
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    return 2 * (precision * recall) / (precision + recall + 1e-10)

In [9]:
import pandas as pd

In [24]:
def evaluate_model(model, dataloader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Set to evaluation mode

    total_loss = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    criterion = torch.nn.BCEWithLogitsLoss()  # Same loss as training
    num_samples = 0
    all_matrics = []

    with torch.no_grad():  # No gradient computation
        for sample in tqdm(dataloader):
            inputs = sample["image"].to(device)
            masks = sample["mask"].to(device)
            outputs = model(inputs)["out"]  # Shape: (batch, 1, H, W)
            loss = criterion(outputs, masks)
            total_loss += loss.item() * inputs.size(0)

            # Convert logits to binary predictions
            
            preds = torch.sigmoid(outputs) > 0.5  # Threshold at 0.5
            preds_ = preds
            masks_ = masks
            preds = preds.cpu().numpy().astype(np.uint8)
            masks = masks.cpu().numpy().astype(np.uint8)

            # Compute metrics per batch
            for i in range(inputs.size(0)):
                all_matrics.append(compute_metrics(preds_[i], masks_[i]))
                # Compute IoU and F1 score
                total_iou += iou_score(masks[i], preds[i])
                total_f1 += f1_score(masks[i], preds[i])
            num_samples += inputs.size(0)

    avg_loss = total_loss / num_samples
    avg_iou = total_iou / num_samples
    avg_f1 = total_f1 / num_samples
    
    df = pd.DataFrame(all_matrics, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall","region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
    df.to_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/baselineModels/deeplabv3/metrics_pretrain.csv", index=False)
    return avg_loss, avg_iou, avg_f1, all_matrics

In [25]:
data_dir = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/gandhinagar_dataset/test"  
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
])

In [26]:
dataset = SegmentationDataset(data_dir, transforms=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

In [27]:
model = torch.load("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/baselineModels/deeplabv3/deeplab_rooftop_full_50_indian_usa.pth", weights_only=False)
print("Model loaded successfully")

Model loaded successfully


In [28]:
avg_loss, avg_iou, avg_f1, matrixs_array = evaluate_model(model, dataloader)

100%|██████████| 16/16 [00:11<00:00,  1.38it/s]


In [29]:
matrixs_array

[[np.float64(0.7799883939589822),
  0.8763971682131723,
  0.9428720474243164,
  0.8305169588630738,
  0.9276429014475788,
  0.7799883939589822,
  0.8763971682131723,
  0.8305169588630738,
  0.9276429014475788,
  1.0],
 [np.float64(0.8320793307994725),
  0.9083442150252025,
  0.9607734680175781,
  0.8825997713573062,
  0.9356356558543505,
  0.8320793307994725,
  0.9083442150252025,
  0.8825997713573062,
  0.9356356558543505,
  1.0],
 [np.float64(0.8627970465717408),
  0.9263457317152369,
  0.9675521850585938,
  0.9167287933708664,
  0.9361665820746626,
  0.8627970465717408,
  0.9263457317152369,
  0.9167287933708664,
  0.9361665820746626,
  1.0],
 [np.float64(0.7976677677282631),
  0.8874473715866715,
  0.9643077850341797,
  0.860247089209816,
  0.9164239175667535,
  0.7976677677282631,
  0.8874473715866715,
  0.860247089209816,
  0.9164239175667535,
  1.0],
 [np.float64(0.8251535449872897),
  0.904201783190834,
  0.9640541076660156,
  0.9443869289373789,
  0.8672969375466243,
  0.82515

In [30]:
import pandas as pd
metrics_df = pd.read_csv("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/Solar_rooftop_Detection_indian_images/baselineModels/deeplabv3/metrics_pretrain.csv")
metrics_df.head(5)
metrics_df.mean()

pixel_iou                  0.788196
pixel_dice                 0.879128
pixel_accuracy             0.959543
pixel_precision            0.876818
pixel_recall               0.890971
region_iou                 0.788196
region_dice                0.879128
region_precision           0.876818
region_recall              0.890971
region_success_accuracy    0.983607
dtype: float64

In [31]:
avg_loss, avg_iou, avg_f1

(0.3004911483311262,
 np.float64(0.7881964422657025),
 np.float64(0.8791281367589262))

avg_loss, avg_iou, avg_f1 = (0.25739510430664314, (0.7829049226583472), (0.8756252143819642))

avg_loss, avg_iou, avg_f1 = (0.3004911483311262), (0.7881964422657025), (0.8791281367589262)